# 5. pandas na prática

Você já sabe ler uma tabela e filtrar. Esta aula fecha o vocabulário do dia a dia:
**selecionar com precisão**, **atribuir sem quebrar nada** e **agrupar para resumir**.

Nada aqui é difícil. É vocabulário, e vale mais praticar do que entender.

> Roda em qualquer lugar, inclusive no Colab: a tabela de exemplo é escrita à mão na
> célula abaixo, e o dataset do exercício vem do próprio seaborn. Nenhum arquivo para
> baixar ou subir.

In [ ]:
import pandas as pd
import seaborn as sns

pd.set_option("display.width", 120)

# a tabela de exemplo, escrita à mão para caber na tela.
# Repare nos dois `None` em `valor`: eles viram NaN, e a gente usa isso na seção 6.
df = pd.DataFrame({
    "vendedor": ["Ana", "Ana", "Bruno", "Bruno", "Carla",
                 "Carla", "Diego", "Diego", "Elena", "Elena"],
    "produto": ["notebook", "cadeira", "mouse", "mesa", "fone",
                "monitor", "livro", "camiseta", "notebook", "caneca"],
    "categoria": ["eletrônicos", "móveis", "eletrônicos", "móveis", "eletrônicos",
                  "eletrônicos", "livros", "roupas", "eletrônicos", "casa"],
    "valor": [3200.0, 890.0, 150.0, None, 250.0, None, 120.5, 79.9, 3100.0, 35.0],
    "qtd": [1, 2, 3, 1, 4, 1, 6, 2, 1, 10],
    "mes": ["jan", "jan", "jan", "fev", "fev", "fev", "mar", "mar", "mar", "mar"],
    "status": ["aprovada", "aprovada", "aprovada", "pendente", "cancelada",
               "aprovada", "aprovada", "pendente", "aprovada", "aprovada"],
})

print("pandas:", pd.__version__)
df

In [ ]:
df[["qtd","mes"]]

---
## 1. Por que `[]` não basta

O colchete resolve o caso simples: `df["valor"]` pega uma coluna e `df[mascara]` filtra
linhas. Mas ele não faz as duas coisas ao mesmo tempo, e tem uma ambiguidade séria quando
o índice é de inteiros. Olhe:

In [ ]:
s = pd.Series([10, 20, 30], index=[2, 0, 1])     # índice de inteiros, fora de ordem
print(s.to_dict())
print("s[0]      ->", s[0], "  <- rótulo 0, não posição 0")
print("s.iloc[0] ->", s.iloc[0], "  <- agora sim, posição 0")
s

In [ ]:
s_2 = s.copy()

s_2.index = s.index.sort_values()
display(s)
display(s_2)

In [ ]:
s_2.index +=1
s_2.index

Índice de inteiros **não é posição**. E isso não é caso raro: depois de qualquer
filtro o índice fica com buracos, e os rótulos deixam de coincidir com as posições. Daí os
dois acessores:

| | significa | fatia |
|---|---|---|
| `.loc` | **rótulo** | inclui o fim |
| `.iloc` | **posição** | exclui o fim, como toda fatia de Python |

---
## 2. `.loc` e `.iloc`

In [ ]:
print(df.loc[0, "vendedor"], "|", df.iloc[0, 0])          # uma célula
print(df.loc[0:2, "vendedor"].tolist(), "<- 3 linhas, o fim ENTRA")
print(df.iloc[0:2, 0].tolist(), "<- 2 linhas, o fim NÃO entra")

In [ ]:
# linhas e colunas ao mesmo tempo: é isso que o [] não faz
display(df.loc[2:4, ["vendedor", "valor"]])
print()
display(df.iloc[2:4, [0, 3]])
print()
print(df.loc[:, "vendedor":"valor"].columns.tolist(), "<- fatia de colunas por nome")

In [ ]:
# máscara também entra no .loc, e aí dá para escolher a coluna junto
caras = df["valor"] > 500
display(df.loc[caras, ["produto", "valor"]])
print()
print(df.loc[caras & (df["status"] == "aprovada"), "produto"].tolist())

> ⚠️ `&` e `|`, nunca `and`/`or`, e sempre com parênteses, porque `&` tem precedência maior que `>`.

In [ ]:
display(df.head(2))

,vendedor,produto,categoria,valor,qtd,mes,status
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada
1,Ana,cadeira,móveis,890.0,2,jan,aprovada


**✏️ Exercícios rápidos**


1. Com `.loc`, pegue as linhas de rótulo 1 a 3 (as três) só com `produto` e `valor`.
2. Com `.iloc`, pegue as duas primeiras linhas e as duas últimas colunas.

In [ ]:
# 1)

display(df.loc[1:3][["produto","valor"]])

# 2)
display(df.iloc[0:2,-2:])


,produto,valor
1,cadeira,890.0
2,mouse,150.0
3,mesa,NaN


,mes,status
0,jan,aprovada
1,jan,aprovada


In [ ]:
df.head(2)

,vendedor,produto,categoria,valor,qtd,mes,status
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada
1,Ana,cadeira,móveis,890.0,2,jan,aprovada


**✏️ Exercícios rápidos**

1. As vendas `aprovada` **e** com `valor` acima de 500.
2. As de `eletrônicos` **ou** `móveis`, de duas formas: com `|` e com `isin`.
3. As que **não** são `aprovada`. Faça com `~` e com `!=`.
4. Tire os parênteses de `df[(df["valor"] > 500) & (df["qtd"] > 1)]` e leia o erro.
   Por que ele acontece?

In [ ]:
#1.

aprovados = df["status"] == "aprovada"
valor_500 = df["valor"] > 500

df[aprovados & valor_500]

,vendedor,produto,categoria,valor,qtd,mes,status
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada
1,Ana,cadeira,móveis,890.0,2,jan,aprovada
8,Elena,notebook,eletrônicos,3100.0,1,mar,aprovada


In [ ]:
eletronicos = df["categoria"] == "eletrônicos"
moveis = df["categoria"] == "móveis"

df[eletronicos | moveis]

,vendedor,produto,categoria,valor,qtd,mes,status
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada
1,Ana,cadeira,móveis,890.0,2,jan,aprovada
2,Bruno,mouse,eletrônicos,150.0,3,jan,aprovada
3,Bruno,mesa,móveis,NaN,1,fev,pendente
4,Carla,fone,eletrônicos,250.0,4,fev,cancelada
5,Carla,monitor,eletrônicos,NaN,1,fev,aprovada
8,Elena,notebook,eletrônicos,3100.0,1,mar,aprovada


In [ ]:
eletronicos_ou_moveis = df["categoria"].isin(["eletrônicos", "móveis"])
df[eletronicos_ou_moveis]

,vendedor,produto,categoria,valor,qtd,mes,status
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada
1,Ana,cadeira,móveis,890.0,2,jan,aprovada
2,Bruno,mouse,eletrônicos,150.0,3,jan,aprovada
3,Bruno,mesa,móveis,NaN,1,fev,pendente
4,Carla,fone,eletrônicos,250.0,4,fev,cancelada
5,Carla,monitor,eletrônicos,NaN,1,fev,aprovada
8,Elena,notebook,eletrônicos,3100.0,1,mar,aprovada


In [ ]:
aprovada = df["status"] == "aprovada"
n_aprovada = df["status"] != "aprovada"

display(df[~aprovada][["produto", "status"]])
display(df[n_aprovada][["produto", "status"]])

,produto,status
3,mesa,pendente
4,fone,cancelada
7,camiseta,pendente


,produto,status
3,mesa,pendente
4,fone,cancelada
7,camiseta,pendente


In [ ]:
# 1)
display(df[(df["status"] == "aprovada") & (df["valor"] > 500)][["produto", "status", "valor"]])

# 2)
display(df[(df["categoria"] == "eletrônicos") | (df["categoria"] == "móveis")][["produto", "categoria"]])
display(df[df["categoria"].isin(["eletrônicos", "móveis"])][["produto", "categoria"]])

# 3)
aprovada = df["status"] == "aprovada"
n_aprovada = df["status"] != "aprovada"

display(df[~aprovada][["produto", "status"]])
display(df[aprovada][["produto", "status"]])
# o ~ inverte a máscara INTEIRA, e vale a pena quando a condição é composta e
# reescrever cada comparação negada ficaria ilegível

# 4)

df[df["valor"] > 500 & df["qtd"] > 1]

# `&` tem precedência MAIOR que `>`. Sem parênteses o Python avalia
# `500 & df["qtd"]` primeiro, e sobra uma Series inteira num `if` implícito,
# que é justamente o que o pandas se recusa a resolver.

,produto,status,valor
0,notebook,aprovada,3200.0
1,cadeira,aprovada,890.0
8,notebook,aprovada,3100.0


,produto,categoria
0,notebook,eletrônicos
1,cadeira,móveis
2,mouse,eletrônicos
3,mesa,móveis
4,fone,eletrônicos
5,monitor,eletrônicos
8,notebook,eletrônicos


,produto,categoria
0,notebook,eletrônicos
1,cadeira,móveis
2,mouse,eletrônicos
3,mesa,móveis
4,fone,eletrônicos
5,monitor,eletrônicos
8,notebook,eletrônicos


,produto,status
3,mesa,pendente
4,fone,cancelada
7,camiseta,pendente


,produto,status
0,notebook,aprovada
1,cadeira,aprovada
2,mouse,aprovada
5,monitor,aprovada
6,livro,aprovada
8,notebook,aprovada
9,caneca,aprovada


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

---
## 3. Atribuir valores

Para **alterar** valores, use `.loc[linhas, coluna]`. É a única forma que o pandas garante.

In [ ]:
d = df.copy()
d.loc[d["status"] == "cancelada", "valor"] = 0
display(d[["produto", "status", "valor"]])

,produto,status,valor
0,notebook,aprovada,3200.0
1,cadeira,aprovada,890.0
2,mouse,aprovada,150.0
3,mesa,pendente,NaN
4,fone,cancelada,0.0
5,monitor,aprovada,NaN
6,livro,aprovada,120.5
7,camiseta,pendente,79.9
8,notebook,aprovada,3100.0
9,caneca,aprovada,35.0


In [ ]:
# E o que acontece se você fizer do jeito "errado":
recorte = df[df["valor"] > 500]
recorte["valor"] = 0

print(df["valor"].tolist())
print("o original mudou?", 0 in df["valor"].tolist())

[3200.0, 890.0, 150.0, nan, 250.0, nan, 120.5, 79.9, 3100.0, 35.0]
o original mudou? False


/tmp/ipykernel_806/1380135313.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recorte["valor"] = 0


O recorte é uma cópia: alterar ele **não toca** no original, e o pandas não avisa nada. Se você queria mudar o `df`, use `.loc`.

### O certo, no mesmo caso

```
df.loc[ quais linhas , qual coluna ] = valor
        └─ a máscara   └─ o nome
```

A máscara escolhe as **linhas**, o nome escolhe a **coluna**, e as duas coisas acontecem
numa **indexação só**. É exatamente isso que faz o pandas escrever no original em vez de
numa cópia.

In [ ]:
d = df.copy()
caras = d["valor"] > 500          # a máscara: quais linhas

d.loc[caras, "valor"] = 0         # zera SÓ essas linhas, e só nessa coluna
d[["produto", "valor"]]

As linhas que não passam no filtro ficam **intactas**, inclusive as que têm nulo,
porque `NaN > 500` é `False`.

In [ ]:
# o lado direito pode ser uma conta: 10% de desconto só nas caras
d = df.copy()
d.loc[d["valor"] > 500, "valor"] = d["valor"] * 0.9

d[["produto", "valor"]]

Repare que o lado direito é a coluna **inteira**, e mesmo assim só as linhas da
máscara foram alteradas. O pandas alinha os dois lados pelo índice antes de escrever.

In [ ]:
# várias colunas de uma vez: passe uma lista
d = df.copy()
d.loc[d["valor"] > 500, ["valor", "qtd"]] = 0

d[["produto", "valor", "qtd"]].head(3)

In [ ]:
display(d)

,vendedor,produto,categoria,valor,qtd,mes,status
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada
1,Ana,cadeira,móveis,890.0,2,jan,aprovada
2,Bruno,mouse,eletrônicos,150.0,3,jan,aprovada
3,Bruno,mesa,móveis,NaN,1,fev,pendente
4,Carla,fone,eletrônicos,0.0,4,fev,cancelada
5,Carla,monitor,eletrônicos,NaN,1,fev,aprovada
6,Diego,livro,livros,120.5,6,mar,aprovada
7,Diego,camiseta,roupas,79.9,2,mar,pendente
8,Elena,notebook,eletrônicos,3100.0,1,mar,aprovada
9,Elena,caneca,casa,35.0,10,mar,aprovada


In [ ]:
# e dá para CRIAR coluna só nas linhas que casam
d = df.copy()
d.loc[d["valor"] > 500, "faixa"] = "alto"
d.loc[~(d["valor"] > 500),"faixa"] = "baixo"
d

,vendedor,produto,categoria,valor,qtd,mes,status,faixa
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada,alto
1,Ana,cadeira,móveis,890.0,2,jan,aprovada,alto
2,Bruno,mouse,eletrônicos,150.0,3,jan,aprovada,baixo
3,Bruno,mesa,móveis,NaN,1,fev,pendente,baixo
4,Carla,fone,eletrônicos,250.0,4,fev,cancelada,baixo
5,Carla,monitor,eletrônicos,NaN,1,fev,aprovada,baixo
6,Diego,livro,livros,120.5,6,mar,aprovada,baixo
7,Diego,camiseta,roupas,79.9,2,mar,pendente,baixo
8,Elena,notebook,eletrônicos,3100.0,1,mar,aprovada,alto
9,Elena,caneca,casa,35.0,10,mar,aprovada,baixo


> ⚠️ As linhas que não casaram ficaram com `NaN`, porque nunca receberam valor. Se você
> quer a coluna inteira preenchida, faça o outro lado também
> (`d.loc[d["valor"] <= 500, "faixa"] = "baixo"`), ou use `np.where`, que resolve os dois
> casos de uma vez. Ele é do NumPy, e a gente volta nele na aula 6.

> **Sobre o jeito errado da célula acima.** Se você encadear tudo numa expressão só,
> `df[df["valor"] > 500]["valor"] = 0`: o pandas levanta `ChainedAssignmentError` e
> aponta o `.loc` como saída. Guardando o recorte numa variável antes, como fizemos, o
> aviso **não** aparece, mas o efeito é o mesmo, nada muda no original.

In [ ]:
d = df.copy()
d.head(2)

,vendedor,produto,categoria,valor,qtd,mes,status
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada
1,Ana,cadeira,móveis,890.0,2,jan,aprovada


**✏️ Exercícios rápidos**


1. Zere o `valor` das vendas com status `pendente`, **sem alterar** o `df` original.
2. Crie `d = df[df["qtd"] > 1]` e faça `d["valor"] = 0`. O `df` mudou? E o `d`?
3. Agora faça a alteração do item 1 direto no `df`, com `.loc`, e confira.

In [ ]:
pendente = d["status"] == "pendente"
d.loc[pendente, "valor"] = 0

d

,vendedor,produto,categoria,valor,qtd,mes,status
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada
1,Ana,cadeira,móveis,890.0,2,jan,aprovada
2,Bruno,mouse,eletrônicos,150.0,3,jan,aprovada
3,Bruno,mesa,móveis,0.0,1,fev,pendente
4,Carla,fone,eletrônicos,250.0,4,fev,cancelada
5,Carla,monitor,eletrônicos,NaN,1,fev,aprovada
6,Diego,livro,livros,120.5,6,mar,aprovada
7,Diego,camiseta,roupas,0.0,2,mar,pendente
8,Elena,notebook,eletrônicos,3100.0,1,mar,aprovada
9,Elena,caneca,casa,35.0,10,mar,aprovada


In [ ]:
# 1)
d = df.copy()
d.loc[d["status"] == "pendente", "valor"] = 0
display(d[["produto", "status", "valor"]])

# 2)
d2 = df[df["qtd"] > 1]
d2["valor"] = 0
print("o df mudou? ", (df["valor"] == 0).any())
print("o d2 mudou? ", (d2["valor"] == 0).all())
# o recorte é uma cópia: a alteração vive só no d2, e o df fica intacto

# 3)  num clone, para não estragar o df que o resto do notebook usa
alvo = df.copy()
alvo.loc[alvo["status"] == "pendente", "valor"] = 0
display(alvo.loc[alvo["status"] == "pendente", ["produto", "status", "valor"]])

<div style="padding: 15px; border-left: 6px solid #0000FF; background-color: #ADD8E6; margin-bottom: 15px;">
  <strong>SIMD:</strong> Single Instruction Multiple Data é uma tecnica de paralelização, muito utilizada em otimização de instruções, como por exemplo em programação para GPU, e é o que as chamadas de mascaras e afim fazem.
</div>


---
## 4. `query`: a mesma máscara, mais legível

Vocês já sabem fazer a máscara. O `query` é açúcar: vale quando a condição fica longa.

In [ ]:

valor_min = 200
print(df[(df["valor"] > valor_min) & (df["status"] == "aprovada")]["produto"].tolist())
print(df.query("valor > 200 and status == 'aprovada'")["produto"].tolist())

minimo = 200                                    # @ pega variável de fora
print(df.query(f"valor > {minimo}")["produto"].tolist())
print(df.query("valor > @minimo")["produto"].tolist())
print(df.query("categoria in ['móveis', 'livros']")["produto"].tolist())

['notebook', 'cadeira', 'notebook']
['notebook', 'cadeira', 'notebook']
['notebook', 'cadeira', 'fone', 'notebook']
['notebook', 'cadeira', 'fone', 'notebook']
['cadeira', 'mesa', 'livro']


Não é mais rápido nem mais poderoso. É mais curto, e não tem parênteses para errar.
Repare que `query` **descarta as linhas com nulo** na comparação, igual à máscara.

**✏️ Exercícios rápidos**


1. Reescreva `df[(df["valor"] > 200) & (df["qtd"] >= 2)]` usando `query`.
2. Guarde `minimo = 500` e filtre com `@minimo` dentro do `query`.
3. Com `query` e `in`, pegue as linhas de `eletrônicos` e `móveis` de uma vez.

In [ ]:
df[(df["valor"] > 200) & (df["qtd"] >= 2)]

,vendedor,produto,categoria,valor,qtd,mes,status
1,Ana,cadeira,móveis,890.0,2,jan,aprovada
4,Carla,fone,eletrônicos,250.0,4,fev,cancelada


In [ ]:
df.query("valor > 200 and qtd >= 2")

,vendedor,produto,categoria,valor,qtd,mes,status
1,Ana,cadeira,móveis,890.0,2,jan,aprovada
4,Carla,fone,eletrônicos,250.0,4,fev,cancelada


In [ ]:
# 1)
#display(df.query("valor > 200 and qtd >= 2")[["produto", "valor", "qtd"]])

# 2)
#minimo = 500
#display(df.query("valor > @minimo"))
#df[df["valor"] > minimo]
# 3)
#display(df.query("categoria in ['eletrônicos', 'móveis']"))
df[df["categoria"].isin(['eletrônicos', 'móveis'])]

,vendedor,produto,categoria,valor,qtd,mes,status
0,Ana,notebook,eletrônicos,3200.0,1,jan,aprovada
1,Ana,cadeira,móveis,890.0,2,jan,aprovada
2,Bruno,mouse,eletrônicos,150.0,3,jan,aprovada
3,Bruno,mesa,móveis,NaN,1,fev,pendente
4,Carla,fone,eletrônicos,250.0,4,fev,cancelada
5,Carla,monitor,eletrônicos,NaN,1,fev,aprovada
8,Elena,notebook,eletrônicos,3100.0,1,mar,aprovada


---
## 5. Agregar: o `groupby` de verdade

Você já sabe `df.groupby("x")["y"].sum()`. Na prática quase nunca é uma agregação só,
você quer contagem, soma e média na mesma tabela, e com nome decente.

In [ ]:
# várias de uma vez: passe uma LISTA
df.groupby("vendedor")["valor"].agg(["count", "sum", "mean"]).round(1)

Funciona, mas os nomes das colunas vêm do nome da função. Quando você agrega colunas
**diferentes**, isso vira uma bagunça.

A forma boa é a **agregação nomeada**: `nome_da_saida=("coluna", "função")`.

In [ ]:
df.groupby("categoria").agg(
    total=("valor", "sum"),
    vendas=("produto", "count"),
    itens=("qtd", "sum"),
    ticket=("valor", "mean"),
).round(1)

Cada linha diz: *a coluna `total` sai de somar `valor`*. Lê-se de uma vez, e o
resultado já vem com nome de relatório.

In [ ]:
# duas chaves: o índice vira hierárquico
df.groupby(["vendedor", "status"])["valor"].sum()

In [ ]:
# ...e reset_index() devolve isso para colunas normais
df.groupby(["vendedor", "status"], as_index=False)["valor"].sum()

### `size` × `count`: a diferença é o nulo

`size` conta **linhas** do grupo. `count` conta **valores não nulos** da coluna. Quando há
nulo, os dois discordam, e é justamente aí que o número engana.

In [ ]:
comparacao = pd.DataFrame({
    "size": df.groupby("vendedor").size(),
    "count": df.groupby("vendedor")["valor"].count(),
})
comparacao

O Bruno tem **2 linhas** mas só **1 valor**: a venda da mesa está com `valor` nulo.
Se você reportar "Bruno fez 1 venda", está errado. Se reportar a média dele dividindo por
2, também.

In [ ]:
# agg funciona com função sua também
df.groupby("vendedor")["valor"].agg(
    amplitude=lambda s: s.max() - s.min(),
    maior="max",
).round(1)

### `transform`: agregar sem encolher

`agg` **reduz**: N grupos viram N linhas. `transform` devolve do **tamanho do original**,
repetindo o valor do grupo em cada linha. É o que serve para calcular percentual dentro
do grupo.

In [ ]:
d = df.copy()
d["total_vendedor"] = d.groupby("vendedor")["valor"].transform("sum")
d["pct_do_vendedor"] = (d["valor"] / d["total_vendedor"] * 100).round(1)

d[["vendedor", "produto", "valor", "total_vendedor", "pct_do_vendedor"]]

Repare no `notebook` da Ana: 78,2% de tudo o que ela vendeu. Sem `transform` você
precisaria agregar, colar o resultado de volta linha a linha e alinhar na mão.

**✏️ Exercícios rápidos**

1. Uma tabela por `mes` com: faturamento, número de vendas e ticket médio, usando
   agregação nomeada.
2. Agrupe por `status` **e** `categoria` e some o `valor`. Qual combinação fatura mais?
3. Crie uma coluna com o percentual que cada venda representa da **categoria** dela
   (use `transform`).

In [ ]:
# 1)
display(df.groupby("mes").agg(
    faturamento=("valor", "sum"),
    vendas=("produto", "count"),
    ticket=("valor", "mean"),
).round(2))

# 2)
por = df.groupby(["status", "categoria"])["valor"].sum().sort_values(ascending=False)
display(por)
print("maior:", por.idxmax(), "->", por.max())

# 3)
d = df.copy()
d["total_categoria"] = d.groupby("categoria")["valor"].transform("sum")
d["pct_da_categoria"] = (d["valor"] / d["total_categoria"] * 100).round(1)
display(d[["produto", "categoria", "valor", "pct_da_categoria"]])

---
## 6. Nulos

Dado real vem com buraco. O que decide a qualidade da sua análise não é *ter* nulo: é
**o que você faz** com ele.

In [ ]:
df.isna().sum()          # onde estão, e quantos

In [ ]:
df[df["valor"].isna()]   # quem são as linhas

### As agregações já pulam o nulo

Esta é a parte que engana: `sum` e `mean` **ignoram** o nulo em silêncio.

In [ ]:
print("soma  :", df["valor"].sum())
print("média :", round(df["valor"].mean(), 2), " <- dividiu por 8, não por 10")
print("count :", df["valor"].count(), "não nulos")
print("len   :", len(df), "linhas")

> ⚠️ A média foi calculada sobre 8 vendas, mas a tabela tem 10. Se o leitor do
> relatório achar que são 10, o número está mentindo para ele. **Diga sempre sobre quantos
> a conta foi feita.**

In [ ]:
# e o que acontece se você "resolver" o nulo com zero
print("média ignorando nulos :", round(df["valor"].mean(), 2))
print("média com fillna(0)   :", round(df["valor"].fillna(0).mean(), 2), " <- outra história")

Nenhuma das duas é certa por natureza. Zero significa *"a venda foi de R$ 0"*; nulo
significa *"não sabemos quanto foi"*. **A escolha é sua, e precisa ser consciente.**

### Preencher e descartar

In [ ]:
d = df.copy()

print("fillna(0)          :", d["valor"].fillna(0).tolist())
print("fillna(média)      :", d["valor"].fillna(d["valor"].mean()).round(1).tolist())
print("ffill (repete acima):", d["valor"].ffill().tolist())

In [ ]:
print("linhas originais            :", len(df))
print("dropna() sem argumento :", len(df.dropna()))
print("dropna(subset=['valor'])    :", len(df.dropna(subset=["valor"])))
print("dropna(how='all')           :", len(df.dropna(how="all")), " <- só se a linha inteira for nula")

`dropna()` sem argumento é agressivo: basta **uma** coluna nula para a linha inteira
sair. Em tabela larga isso apaga metade do dado sem você perceber. Quase sempre você quer
`subset=`.

**✏️ Exercícios rápidos**

1. Quantas linhas sobram com `dropna()` e com `dropna(subset=["valor"])`? Por que a
   diferença é essa neste `df`?
2. Preencha o `valor` nulo com a **média da categoria** da linha (dica: `transform`).
3. Calcule o ticket médio duas vezes: ignorando nulos e com `fillna(0)`. Qual você
   colocaria num relatório, e o que escreveria embaixo do número?

In [ ]:
# 1)
print("dropna()              ->", len(df.dropna()), "linhas")
print("dropna(subset=valor)  ->", len(df.dropna(subset=["valor"])), "linhas")
# Iguais: neste df o ÚNICO nulo está em `valor`. Numa tabela com nulo em várias
# colunas o dropna() sem subset seria bem mais agressivo.

# 2)
d = df.copy()
d["valor"] = d["valor"].fillna(d.groupby("categoria")["valor"].transform("mean"))
display(d[["produto", "categoria", "valor"]].round(2))

# 3)
print("ignorando nulos:", round(df["valor"].mean(), 2))
print("com fillna(0)  :", round(df["valor"].fillna(0).mean(), 2))
# No relatório eu poria a primeira, com a nota: "média de 8 vendas; 2 sem valor
# registrado". O zero afirmaria que a venda foi de R$ 0, que não é o que sabemos.

---
## Exercício em aula

Dataset novo, e o pandas é o mesmo. O `penguins` tem 344 pinguins com espécie, ilha,
medidas do bico e da nadadeira, massa e sexo. Ele vem do seaborn, então no Colab basta
a linha abaixo.

In [ ]:
pinguins = sns.load_dataset("penguins")
pinguins.head()

Responda usando só o que vimos hoje:

1. Quantas linhas e colunas? Quais colunas têm nulo, e quantos em cada?
2. Com `.loc`, mostre só `species`, `island` e `body_mass_g` das 5 primeiras linhas.
3. Quantos pinguins de cada `species`? E de cada `island`?
4. Com `query`, quantos pinguins passam de 5000 g? ATE AQUI
5. Agrupe por `species` e traga: contagem, massa média e comprimento médio do bico.
6. No mesmo agrupamento, traga `size` e `count` de `body_mass_g`. De onde vem a diferença?
7. Descarte as linhas sem `body_mass_g`. Quantas sobraram?
8. Crie a coluna `massa_kg` e agrupe por `species` **e** `sex` ao mesmo tempo.

In [ ]:
# 1)
print(pinguins.shape)
print("nulos:", pinguins.isna().sum().to_dict())
# 344 linhas e 7 colunas. Os nulos estão nas 4 medidas (2 cada) e em `sex` (11).
# São 11 linhas no total: as 2 sem medida também não têm sexo

# 2)
display(pinguins.loc[:4, ["species", "island", "body_mass_g"]])
# no .loc o 4 ENTRA: rótulo é inclusivo dos dois lados

# 3)
print(pinguins["species"].value_counts().to_dict())
print(pinguins["island"].value_counts().to_dict())

# 4)
pesados = pinguins.query("body_mass_g > 5000")
print(f"\n{len(pesados)} pinguins acima de 5000 g")
print("e todos são da mesma espécie:", pesados["species"].unique().tolist())

# 5)
display(pinguins.groupby("species").agg(
    pinguins=("body_mass_g", "size"),
    massa_media=("body_mass_g", "mean"),
    bico_medio=("bill_length_mm", "mean"),
).round(1))
# o Gentoo pesa 5076 g contra 3700 do Adelie: é ele que enche o grupo dos >5000

# 6)
display(pinguins.groupby("species").agg(
    linhas=("body_mass_g", "size"),
    com_valor=("body_mass_g", "count"),
))
# a diferença são os 2 nulos: um Adelie (152 -> 151) e um Gentoo (124 -> 123).
# `size` conta linhas, `count` conta valores preenchidos, e toda média usa o count

# 7)
com_massa = pinguins.dropna(subset=["body_mass_g"])
print(len(pinguins), "->", len(com_massa), "linhas")

# 8)
com_massa = com_massa.copy()
com_massa["massa_kg"] = com_massa["body_mass_g"] / 1000

display(com_massa.groupby(["species", "sex"]).agg(
    pinguins=("massa_kg", "size"),
    massa_kg=("massa_kg", "mean"),
).round(2))
# o macho é mais pesado que a fêmea nas três espécies, e a distância é parecida:
# o sexo explica uma parte, e a espécie explica outra

## Para casa

1. Qual espécie tem a maior massa média? E dentro dela, macho ou fêmea pesa mais?
2. Use `transform` para criar `vs_especie`: a massa do pinguim dividida pela massa média
   da espécie dele. Quem é o maior fora da curva?
3. As 11 linhas sem `sex` mudam alguma conclusão do item 1? Compare a massa média por
   espécie com e sem elas.

In [ ]:
# 1)
por_especie = pinguins.groupby("species")["body_mass_g"].mean().round(1)
print(por_especie.to_dict())
print("maior massa média:", por_especie.idxmax())

gentoo = pinguins.query("species == 'Gentoo'")
print("\nGentoo por sexo:",
      gentoo.groupby("sex")["body_mass_g"].mean().round(0).to_dict())
# o Gentoo, com 5076 g. Dentro dele o macho pesa 5485 contra 4680 da fêmea

# 2)
com_massa = pinguins.dropna(subset=["body_mass_g"]).copy()
com_massa["vs_especie"] = (com_massa["body_mass_g"]
                           / com_massa.groupby("species")["body_mass_g"].transform("mean"))

display(com_massa.nlargest(3, "vs_especie")[
    ["species", "sex", "body_mass_g", "vs_especie"]].round(3))
# um Adelie de 4775 g, 29% acima da média da espécie dele. Em massa absoluta ele nem
# aparece, porque qualquer Gentoo é mais pesado. `transform` é o que deixa comparar
# cada um contra o próprio grupo

# 3)
com_todas = pinguins.groupby("species")["body_mass_g"].mean().round(1)
so_com_sexo = pinguins.dropna(subset=["sex"]).groupby("species")["body_mass_g"].mean().round(1)

display(pd.DataFrame({"com todas": com_todas,
                      "só com sexo": so_com_sexo,
                      "diferença": (so_com_sexo - com_todas).round(1)}))
# não mudam nada: a maior diferença é 16.4 g no Gentoo, contra uma média de 5076.
# O Gentoo continua o mais pesado. Descartar 11 de 344 linhas foi inofensivo AQUI, e
# isso é uma conclusão que se confere, não se supõe

## Dicas

Olhem a documentação do pandas, tem muitas outras funcionalidades interessantes.

[Documentação Pandas](https://pandas.pydata.org/docs/reference/index.html)

Outras bibliotecas interessantes junto ao pandas:

[Polars](https://docs.pola.rs/py-polars/html/reference/) \
[DuckDB](https://duckdb.org/docs/lts/clients/python/overview)